# reFuel to MOTEL unmapped records

This notebook structures the conversion of reFuel technology rows into MOTEL-compatible unmapped records.

## Goals
- Read reFuel input data from Excel.
- Load and inspect the MOTEL schema.
- Map each source row into a normalized record payload.
- Validate required fields and export records for review.

## Output target
- Write records to `dev2/motel-db/records/` as JSON or YAML files.

In [1]:
## --- load packages
from pathlib import Path
import pandas as pd
import yaml

import math
import re
from collections import defaultdict

# 1) Inputs and setup

## 1.1 Source data (reFuel Excel)

In [2]:
path_refuel = Path(r"E:\Barton\repositories\motel-platform\dev2\data\reFuel_TechDatabase_Clean_2026-06-03.xlsx")

refuel_data = pd.read_excel(path_refuel, sheet_name=None)
print(f"sheets found: {list(refuel_data.keys())}")

sheets found: ['Metadata', 'Nomenclature', 'Carrier', 'ConvTech', 'StorTech', 'EmbeddedCarbon', 'Reference']


## 1.2 Load MOTEL schema

In [3]:
path_schema = r'schema\motel_schema_rev2.yaml'

with open(path_schema, "r", encoding="utf-8") as f:
    schema_motel = yaml.safe_load(f)

print(f"Schema loaded successfully")
print(f"Type: {type(schema_motel)}")
if isinstance(schema_motel, dict):
    print(f"Top-level keys ({len(schema_motel)}): {list(schema_motel.keys())[:10]}")

Schema loaded successfully
Type: <class 'dict'>
Top-level keys (13): ['unmapped_entity', 'linked_entity', 'technology', 'process', 'source', 'contributor', 'review', 'attribute', 'carrier', 'geographic_scope']


# 2) Create secondary data from Refuel.ch data

The schema is created and then the first version of the secondary data are produced based on:
 - pre-define MOTEL schema
 - reFuel.ch tech data
 - LLM (Gemini)


For each dataset, there are two files:
1. the actual data in dataset (produce from reFuel.ch so far)
2. the map from reFuel.ch data (name or id) to motel name/id

In [4]:
# initialise the dataset and mapping data
datasets = {}

## 2.1 Technology

Technology = The "How" (The Engineering Asset)
A technology is the specific hardware, design variant, or asset configuration that actually executes a process. A single process can often be performed by several different technologies, each featuring unique efficiencies, investment costs, emission profiles, or commercial scales.

In [5]:
path_tech = r'motel-db\secondary\technologies.csv'

ds_tech = datasets['tech'] = pd.read_csv(path_tech)
ds_tech

,tech_id,technology_name,technology_description,technology_variant,main_process,main_operation_unit,ontology_iri
0,tech_00001,Ammonia CCGT Power Generation,Ammonia-fueled combined cycle gas turbine (CCG...,CCGT,Ammonia Power Generation,Gas turbine,NaN
1,tech_00002,Ammonia OCGT Power Generation,Ammonia-fueled open cycle gas turbine (OCGT) e...,OCGT,Ammonia Power Generation,Gas turbine,NaN
2,tech_00003,Methane CCGT Power Generation,Methane-fueled CCGT with carbon capture and st...,CCGT,Methane Power Generation,Gas turbine,NaN
3,tech_00004,Methane CHP Generation,Combined heat and power (CHP) generation using...,CHP plant,Methane Combined Heat and Power,Combined heat & power,NaN
4,tech_00005,Methane Fuel Cell Power Generation,Methane-fueled fuel cell electricity generatio...,SOFC,Methane Power Generation,Fuel cell,NaN
5,tech_00006,Methane Fuel Cell Generation in Buildings,Methane-fueled fuel cell electricity generatio...,SOFC,Methane Power Generation,Fuel cell,NaN
6,tech_00007,Methane OCGT Power Generation,Methane-fueled open cycle gas turbine (OCGT) e...,OCGT,Methane Power Generation,Gas turbine,NaN
7,tech_00008,Geothermal CHP Generation,Geothermal-based CHP generation (2040).,Geothermal power plant,Geothermal Combined Heat and Power,Combined heat & power,NaN
8,tech_00009,Hydrogen CCGT Power Generation,Hydrogen-fueled combined cycle gas turbine (CC...,CCGT,Hydrogen Power Generation,Gas turbine,NaN
9,tech_00010,Hydrogen Fuel Cell Power Generation,Electricity generation via fuel cells at node ...,SOFC,Hydrogen Power Generation,Fuel cell,NaN


## 2.2 Process

Process = The "What" (The Functional Slot)
A process defines an abstract transformation. It maps out the strict thermodynamic or economic boundaries of the system—specifically, what goes in (feedstocks) and what comes out (products). It does not care about the exact machinery being used.

In [6]:
path_proc = r'motel-db\secondary\processes.csv'

ds_proc = datasets['proc'] = pd.read_csv(path_proc)
ds_proc

,process_id,process_name,process_description,process_type,process_category,main_sector,feedstocks,products
0,proc_00001,Ammonia Power Generation,Conversion of ammonia into network electricity...,conversion,Power Generation,power,['carrier_nh3'],['carrier_elec']
1,proc_00002,Methane Power Generation,Combustion or electrochemical conversion of me...,conversion,Power Generation,power,['carrier_ch4'],['carrier_elec']
2,proc_00003,Methane Combined Heat and Power,Simultaneous co-generation of utility power an...,conversion,Co-generation,power,['carrier_ch4'],"['carrier_elec', 'carrier_heat']"
3,proc_00004,Geothermal Combined Heat and Power,Deep geothermal extraction supplying regional ...,conversion,Co-generation,power,['carrier_geothermal_heat'],"['carrier_elec', 'carrier_heat']"
4,proc_00005,Hydrogen Power Generation,Dispatched power generation utilizing gaseous ...,conversion,Power Generation,power,['carrier_h2'],['carrier_elec']
5,proc_00006,Pumped Hydro Storage,Mechanical load-balancing via water-based pote...,storage,Energy Storage,power,['carrier_elec'],['carrier_elec']
6,proc_00007,Hydroelectric Power Generation,Kinetic and potential hydraulic energy capture...,conversion,Power Generation,power,['carrier_water_flow'],['carrier_elec']
7,proc_00008,Waste Power Generation,Thermal treatment of municipal solid waste for...,conversion,Power Generation,power,['carrier_msw'],['carrier_elec']
8,proc_00009,Solar Photovoltaic Generation,Direct solar irradiance conversion into altern...,conversion,Power Generation,power,['carrier_solar_radiation'],['carrier_elec']
9,proc_00010,Nuclear Power Generation,Controlled uranium-235 fission generating high...,conversion,Power Generation,power,['carrier_u235'],['carrier_elec']


## 2.3 Source

In [7]:
path_src = r'motel-db\secondary\sources.csv'

ds_src = datasets['src'] = pd.read_csv(path_src)
ds_src

,source_id,source_name,description,source_type,link,note,access_date
0,src_00001,Allgöwer PATHFNDER Study,Paper by ETH from PATHFNDER,paper,https://doi.org/10.1021/acs.iecr.4c01287,NaN,NaN
1,src_00002,Empa Rheintal Study,"Rheintal paper of Empa, submitted 2025",paper,https://dx.doi.org/10.2139/ssrn.5291581,NaN,NaN
2,src_00003,Journal of Cleaner Production Biochar Study,Financial feasibility of biochar production: A...,paper,https://doi.org/10.1016/j.jclepro.2025.147167,NaN,NaN
3,src_00004,Future Energy Hydropower Framework,Future Energy (Third Edition),paper,https://doi.org/10.1016/B978-0-08-102886-5.000...,NaN,NaN
4,src_00005,CCGT and CCS Integration Modelling Study,Off-design point modelling of a 420 MW CCGT po...,paper,https://doi.org/10.1016/j.apenergy.2016.06.087,This wasn't used anywhere in the db?,NaN
5,src_00006,Electricity Market Combined Cycle Modelling Study,Modelling combined-cycle power plants in a det...,paper,https://doi.org/10.1016/j.energy.2024.131246,NaN,NaN
6,src_00007,Municipal Solid Waste Gasification Review,Municipal Solid Waste Gasification: Technologi...,paper,https://doi.org/10.3390/su17156704,NaN,NaN
7,src_00008,Renewable Cost Projections Review,"review the cost projection of PV, wind, battery",paper,https://doi.org/10.1016/j.apenergy.2025.125856,NaN,NaN
8,src_00009,RWGS Coupled Catalytic Reactions Study,Technology Readiness and Emerging Prospects of...,paper,https://doi.org/10.1002/cssc.202400865,NaN,NaN
9,src_00010,ETHZ Alpine PV Viability Study,ETHZ paper on Alpine PV cost and financial via...,paper,https://doi.org/10.1016/j.apenergy.2024.124019,NaN,NaN


## 2.4 Attribute

note: exclude attirbutes for balancing, deal with it for balancing later

In [8]:
path_attr = r'motel-db\controlled_vocabulary\attributes.csv'

ds_attr = datasets['attr'] = pd.read_csv(path_attr)
ds_attr

,attribute_id,attribute_name,attribute_description,unit,data_format,note
0,ATTR_TECHNICAL_EFFICIENCY,technical_efficiency,Overall energy conversion efficiency: main out...,-,float,Based on kW output capacity / LHV (chemical) e...
1,ATTR_TECHNOLOGY_READINESS_LEVEL,technology_readiness_level,Technology Readiness Level on the standard 1–9...,-,int,See Nomenclature A-3.
2,ATTR_TECH_MATURITY,tech_maturity,Aggregated maturity classification derived fro...,-,text,See Nomenclature A-2.
3,ATTR_REFERENCE_UNIT_SIZE,reference_unit_size,Nameplate capacity on which all specific cost ...,CAP,float,Capacity basis for scaling.
4,ATTR_THEORETICAL_EFFICIENCY,theoretical_efficiency,Ratio of the main output's energy content (LHV...,-,float,May be blank if not applicable.
5,ATTR_OPERATING_TEMPERATURE,operating_temperature,Typical operating temperature of the process.,°C,float,Relevant for thermal integration.
6,ATTR_POWER_CAPACITY_MIN,power_capacity_min,Smallest assumed unit capacity for the technol...,CAP,float,0 if no minimum constraint.
7,ATTR_LIFETIME_ECONOMIC,lifetime_economic,Economic lifetime of the asset.,yr,int,Expected operational lifetime.
8,ATTR_CAPEX_ONE_TIME,capex_one_time,Fixed capital expenditure (independent of size).,CUR,float,For project lump sum.
9,ATTR_OPEX_FIX_PCT_OF_CAPEX,opex_fix_pct_of_capex,Fixed annual operational expenditure expressed...,%,float,-


## 2.X Mappings from reFuel.ch to MOTEL

In [9]:
mappings = {}

In [10]:
path_map_tech = 'motel-db/mapping/refuel/map_tech.csv'

map_tech = mappings['tech'] = pd.read_csv(path_map_tech)
map_tech

,previous_technology_name,new_technology_name
0,NH3_CCGT_El,Ammonia CCGT Power Generation
1,NH3_OCGT_El,Ammonia OCGT Power Generation
2,CH4_CCGT_El,Methane CCGT Power Generation
3,CH4_CHP_ElTh,Methane CHP Generation
4,CH4_FuelCell_El,Methane Fuel Cell Power Generation
5,CH4_FuelCell_ElBdg,Methane Fuel Cell Generation in Buildings
6,CH4_OCGT_El,Methane OCGT Power Generation
7,Geothermal_CHP_ElTh,Geothermal CHP Generation
8,H2_CCGT_El,Hydrogen CCGT Power Generation
9,H2_FuelCell_El,Hydrogen Fuel Cell Power Generation


In [11]:
path_map_src = 'motel-db/mapping/refuel/map_src.csv'

map_src = mappings['src'] = pd.read_csv(path_map_src)

# dict consumed by parse_source_text: reFuel source_id -> MOTEL source_name
map_src_id = dict(zip(map_src['previous_source_id'], map_src['new_source_name']))
map_src

,previous_source_id,new_source_name
0,Allgoewer_2024,Allgöwer PATHFNDER Study
1,rheintal_2025,Empa Rheintal Study
2,biochar_jcp_2025,Journal of Cleaner Production Biochar Study
3,book_hydropower_2020,Future Energy Hydropower Framework
4,paper_CCGT+CCS_2016,CCGT and CCS Integration Modelling Study
5,paper_CCGT_2024,Electricity Market Combined Cycle Modelling Study
6,paper_Msw to syngas,Municipal Solid Waste Gasification Review
7,paper_PVcostProjection_2025,Renewable Cost Projections Review
8,paper_RWGS,RWGS Coupled Catalytic Reactions Study
9,paper_alpinePV_2024,ETHZ Alpine PV Viability Study


In [12]:
path_map_attr = 'motel-db/mapping/refuel/map_attr.csv'

df_attr_mapping = pd.read_csv(path_map_attr)
df_attr_mapping

,Raw Attribute,Cleaned attribute_name,Assigned Target attribute_id
0,technical_efficiency,technical_efficiency,ATTR_TECHNICAL_EFFICIENCY
1,trl,technology_readiness_level,ATTR_TECHNOLOGY_READINESS_LEVEL
2,tech_maturity,tech_maturity,ATTR_TECH_MATURITY
3,reference_unit_size,reference_unit_size,ATTR_REFERENCE_UNIT_SIZE
4,theoretical_efficiency,theoretical_efficiency,ATTR_THEORETICAL_EFFICIENCY
5,operating_temperature_c,operating_temperature,ATTR_OPERATING_TEMPERATURE
6,min_installation_size,power_capacity_min,ATTR_POWER_CAPACITY_MIN
7,lifetime_yr,lifetime_economic,ATTR_LIFETIME_ECONOMIC
8,capex_one_time,capex_one_time,ATTR_CAPEX_ONE_TIME
9,opex_fix_pct_of_capex,opex_fix_pct_of_capex,ATTR_OPEX_FIX_PCT_OF_CAPEX


## 2.X Data validation

Check if the data in the seconary dataset are correct
1. validate the data format based on schema
2. check the id and name are unique
3. check the link between tech and process are matched

In [13]:
import os
from pathlib import Path
import pandas as pd
import yaml


def validate_csv(df, schema):
    # TODO: how/what is validated here?

    # Skip validation for 'category' and 'description' columns
    schema = {k: v for k, v in schema.items() if k not in ['category', 'description']}
    df_columns = [col for col in df.columns if col not in ['category', 'description']]
    
    # Validate against schema (for simplicity, checking if all keys in schema exist in DataFrame columns)
    matched_columns = set(schema.keys()) & set(df_columns)
    if matched_columns:
        print("Matched Columns:", list(matched_columns))

    missing_keys = set(schema.keys()) - set(df_columns)
    if missing_keys:
        print("Missing Columns:", list(missing_keys))    

    extra_keys = set(df_columns) - set(schema.keys())
    if extra_keys:
        print(f"Extra keys: {extra_keys}\n")

    if missing_keys or extra_keys:
        print("The DataFrame does not match the schema\n")
        return False
    
    print(f"DataFrame follows the schema\n")
    
    return True


# Get the last three CSV files
di_datasets = {
    'technology': ds_tech,
    'process': ds_proc,
    'source': ds_src
    }

# Validate each of the last three CSV files against the schema
for ds_nam, df in di_datasets.items():
    validate_csv(df, schema_motel[ds_nam])

Matched Columns: ['technology_name', 'tech_id', 'main_process', 'technology_description', 'ontology_iri', 'technology_variant', 'main_operation_unit']
DataFrame follows the schema

Matched Columns: ['process_type', 'process_name', 'feedstocks', 'main_sector', 'process_id', 'process_category', 'process_description', 'products']
DataFrame follows the schema

Matched Columns: ['source_id', 'source_type', 'access_date', 'link', 'source_name', 'note']
Missing Columns: ['assessment_method', 'source_description', 'confidence_level', 'assessment_date']
The DataFrame does not match the schema



# 3) Create unmapped recrods from reFuel.ch data

## 3.1 Convert/map the reFuel.ch data to unmapped records

In [14]:
## load the ConvTech
sheet_name = 'ConvTech'
df_convtech = refuel_data[sheet_name].copy()

## correct column names
df_convtech.columns = df_convtech.loc[1]
df_convtech = df_convtech.loc[2:].reset_index(drop=True)

# deal with tech_category not in a column: add tech_category
df_convtech.insert(0, 'tech_category', pd.NA)
indx_filter = df_convtech[df_convtech['tech_type'].isna()].index

df_convtech.loc[indx_filter, 'tech_category'] = df_convtech.loc[indx_filter, 'tech_id']

df_convtech['tech_category'] = df_convtech['tech_category'].ffill(inplace=True)
df_convtech = df_convtech[df_convtech['tech_type'].notna()].reset_index(drop=True)

df_convtech

1,tech_category,tech_id,tech_year,technology_class,description,unit_operation,tech_type,reference_unit_size,cost_base,currency,...,opex_per_energy,discount_rate_pct,value_range_check,uncertainty_rating,mass_inflows,mass_outflows,energy_balance_error_pct,mass_balance_error_pct,balance_pass_flag,list_of_source_id
0,Electricity generation,NH3_CCGT_El,2050,Gas turbine,NaN,CCGT,Conversion,NaN,ECA,EUR,...,0.002,NaN,true,NaN,"1216, 1714",1000,0.6906,0.0122,true,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
1,Electricity generation,NH3_OCGT_El,2050,Gas turbine,NaN,OCGT,Conversion,NaN,ECA,EUR,...,0.002,NaN,true,NaN,"1216, 1714",1000,0.6906,0.0122,true,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
2,Electricity generation,CH4_CCGT_El,2050,Gas turbine,Methane-fueled CCGT with carbon capture and st...,CCGT,Conversion,NaN,ECA,EUR,...,0.002,NaN,true,Low,na,na,0.0600,0.0000,true,"""technical_efficiency"": paper_CCGT_2024; ""cap..."
3,Electricity generation,CH4_CHP_ElTh,2050,Combined heat & power,Combined heat and power (CHP) generation using...,CHP plant,Conversion,NaN,ECA,EUR,...,0.004,NaN,true,Low,na,na,0.0600,0.0000,true,"""capex_per_capacity"", ""opex_per_capacity_yr"",..."
4,Electricity generation,CH4_FuelCell_El,2030,Fuel cell,Methane-fueled fuel cell electricity generatio...,SOFC,Conversion,NaN,ECA,EUR,...,0,NaN,true,Medium,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,Carbon capture & sequestration,Air_DAC_CO2,2050,Carbon capture,Direct air capture using adsorption technology...,DAC (sorbent-based),Conversion,NaN,ECA,EUR,...,0,NaN,true,High,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
95,Carbon capture & sequestration,CO2Flue_PSA_CO2,2030,Carbon capture,CO2 post combustion capture via Hot-Potassium ...,Pressure swing adsorption,Conversion,20,ECA,EUR,...,0.0022 EUR/kg,NaN,true,High,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
96,Carbon capture & sequestration,CO2Flue_PSA_CO2,2040,Carbon capture,CO2 post combustion capture via Hot-Potassium ...,Pressure swing adsorption,Conversion,20,ECA,EUR,...,0.0022 EUR/kg,NaN,true,High,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
97,Carbon capture & sequestration,CO2Flue_PSA_CO2,2050,Carbon capture,CO2 post combustion capture via Hot-Potassium ...,Pressure swing adsorption,Conversion,20,ECA,EUR,...,0.0022 EUR/kg,NaN,true,High,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."


In [15]:
## supportive functions

def is_nan(value):
    return value is None or (
        isinstance(value, float) and math.isnan(value)
    )


def clean(value):
    return None if is_nan(value) else value


def parse_number_with_unit(text):
    """
    Converts:
        '2800 CHF/kg/h'
    into:
        (2800.0, 'CHF/kg/h')
    """
    if not isinstance(text, str):
        return text, None

    match = re.match(r"^\s*([0-9.]+)\s*(.*)$", text)
    if match:
        return float(match.group(1)), match.group(2).strip()

    return text, None


def split_csv(value):
    if is_nan(value):
        return []
    return [
        x.strip()
        for x in str(value).split(",")
        if x.strip() and x.strip().lower() not in {"na", "nan"}
    ]


def split_csv_float(value):
    out = []
    for token in split_csv(value):
        try:
            out.append(float(token))
        except ValueError:
            out.append(None)
    return out


def split_number_unit(value):
    """Split strings like '20000 kg/h' into (20000.0, 'kg/h').

    Returns (original_value, None) when no numeric+unit pattern is found.
    """
    if not isinstance(value, str):
        return value, None

    text = value.strip()
    if not text:
        return value, None

    match = re.match(r"^([-+]?\d[\d,]*(?:\.\d+)?)\s+(.+)$", text)
    if not match:
        return value, None

    number_text, unit_text = match.groups()
    unit_text = unit_text.strip()
    if not unit_text:
        return value, None

    try:
        numeric_value = float(number_text.replace(",", ""))
    except ValueError:
        return value, None

    return numeric_value, unit_text


def is_empty_like_base(value):
    """Return True for empty / NA-like values used for final record pruning."""
    if is_nan(value):
        return True

    if isinstance(value, str):
        token = value.strip().lower()
        return token in {"", "na", "nan", "none", "null", "n/a"}

    if isinstance(value, (list, tuple, set)):
        return len(value) == 0 or all(is_empty_like_base(v) for v in value)

    if isinstance(value, dict):
        return len(value) == 0 or all(is_empty_like_base(v) for v in value.values())

    return False


def is_empty_like(value):
    """Return True for empty / NA-like values.

    Empty-like values include None, NaN, empty strings, and textual NA markers.
    Containers are considered empty when all contained values are empty-like.
    """
    if is_nan(value):
        return True

    if isinstance(value, str):
        token = value.strip().lower()
        return token in {"", "na", "nan", "none", "null", "n/a"}

    if isinstance(value, (list, tuple, set)):
        return len(value) == 0 or all(is_empty_like(v) for v in value)

    if isinstance(value, dict):
        return len(value) == 0 or all(is_empty_like(v) for v in value.values())

    return False

In [16]:
# LAYER 1 (BASE): core row -> record mapping.
# This cell defines the base implementation of map_technology and helpers.
# Balancing is built directly in final target schema here.
## --- function to convert reFuel.ch data into 'unmapped record'


def add_value(values, name, description, value, unit=None, scenario=None, time_index=None):
    """Append one attribute if and only if value is non-empty.

    Also supports overriding default unit when value embeds unit text,
    e.g., value='20000 kg/h' with default unit='kW or kg/h'.
    """
    if is_empty_like(value):
        return

    parsed_value, parsed_unit = split_number_unit(value)
    if parsed_unit:
        value = parsed_value
        unit = parsed_unit

    record = {
        "attribute_name": name,
        "attribute_description": description,
        "unit": unit,
        "value": value,
    }

    if scenario:
        record["scenario"] = scenario

    if time_index:
        record["time_index"] = time_index

    values.append(record)


def build_balancing_entries(carriers, shares, units):
    size = max(len(carriers), len(shares), len(units))
    entries = []

    for i in range(size):
        carrier = carriers[i] if i < len(carriers) else None
        share = shares[i] if i < len(shares) else None
        unit = units[i] if i < len(units) else None

        if carrier is None and share is None and unit is None:
            continue

        entries.append({
            "carrier": carrier,
            "share": share,
            "unit": unit,
        })

    return entries


def to_balance_list(items):
    """Normalize balancing items to list of dicts with carrier_name/share/unit."""
    normalized = []
    for item in items or []:
        if not isinstance(item, dict):
            continue

        carrier_name = clean(item.get("carrier_name"))
        if carrier_name is None:
            carrier_name = clean(item.get("carrier"))

        share = clean(item.get("share"))
        unit = clean(item.get("unit"))

        if carrier_name is None and share is None and unit is None:
            continue

        normalized.append({
            "carrier_name": carrier_name,
            "share": share,
            "unit": unit,
        })

    return normalized


def prune_empty_keys(obj):
    """Recursively remove keys/items that are empty-like."""
    if isinstance(obj, dict):
        cleaned = {}
        for key, value in obj.items():
            pruned = prune_empty_keys(value)
            if not is_empty_like_base(pruned):
                cleaned[key] = pruned
        return cleaned

    if isinstance(obj, list):
        cleaned_list = []
        for value in obj:
            pruned = prune_empty_keys(value)
            if not is_empty_like_base(pruned):
                cleaned_list.append(pruned)
        return cleaned_list

    return obj


def normalize_unit(unit_text):
    if is_nan(unit_text):
        return None
    unit = str(unit_text).strip()
    if not unit or unit == "-":
        return None
    return unit


def split_or_units(unit_text):
    """Split expressions like 'kW or kg/h' into candidate units."""
    if not isinstance(unit_text, str):
        return []
    return [u.strip() for u in re.split(r"\s+or\s+", unit_text, flags=re.IGNORECASE) if u.strip()]


def infer_main_output_unit(main_output_carrier, output_carriers, output_units):
    """Infer the unit for the selected main output carrier."""
    if main_output_carrier is not None:
        for carrier, unit in zip(output_carriers, output_units):
            if clean(carrier) == main_output_carrier:
                unit_norm = normalize_unit(unit)
                if unit_norm:
                    return unit_norm

    # Fallback: first non-empty output unit if main output is not matched.
    for unit in output_units:
        unit_norm = normalize_unit(unit)
        if unit_norm:
            return unit_norm

    return None


def choose_preferred_or_unit(unit_text, main_output_unit):
    """Pick one unit from a disjunctive unit string using output-unit context."""
    candidates = split_or_units(unit_text)
    if not candidates:
        return normalize_unit(unit_text)

    if not main_output_unit:
        return candidates[0]

    target = str(main_output_unit).strip().lower()
    candidates_lower = [c.lower() for c in candidates]

    # Exact match first.
    for idx, cand_lower in enumerate(candidates_lower):
        if cand_lower == target:
            return candidates[idx]

    # Heuristics for common power vs mass-flow disambiguation.
    if ("kg/" in target) or target.startswith("kg") or ("ton/" in target) or target.startswith("t/"):
        for idx, cand_lower in enumerate(candidates_lower):
            if "kg/" in cand_lower or cand_lower.startswith("kg") or "ton/" in cand_lower or cand_lower.startswith("t/"):
                return candidates[idx]

    if ("w" in target) or ("wh" in target) or ("j/s" in target):
        for idx, cand_lower in enumerate(candidates_lower):
            if ("w" in cand_lower) or ("wh" in cand_lower) or ("j/s" in cand_lower):
                return candidates[idx]

    return candidates[0]


def resolve_attribute_unit(attribute_name, unit_text, main_output_unit):
    """Resolve attribute unit; supports context-based choice for 'A or B' units."""
    unit_norm = normalize_unit(unit_text)
    if not unit_norm:
        return None

    # Keep this generic so future attributes with disjunctive units also work.
    if isinstance(unit_norm, str) and re.search(r"\s+or\s+", unit_norm, flags=re.IGNORECASE):
        return choose_preferred_or_unit(unit_norm, main_output_unit)

    return unit_norm


def log_main_output_unit_resolution(tech_name, main_output_carrier, main_output_unit, specs):
    """Print a trace line showing inferred main-output unit and resolved capacity default."""
    capacity_unit_options = None
    for spec in specs:
        if spec.get("attribute_name") == "capacity_min":
            raw_unit = spec.get("unit")
            if isinstance(raw_unit, str) and re.search(r"\s+or\s+", raw_unit, flags=re.IGNORECASE):
                capacity_unit_options = raw_unit
                break

    if capacity_unit_options:
        selected_capacity_unit = choose_preferred_or_unit(capacity_unit_options, main_output_unit)
    else:
        selected_capacity_unit = None

    # print(
    #     f"[unit-resolution] tech '{tech_name}' has main_output '{main_output_carrier}' "
    #     f"with unit '{main_output_unit}', and default capacity unit '{selected_capacity_unit}' is used."
    # )


def build_attribute_specs(mapping_df):
    # new schema does not require description and unit in attribute mapping
    required_cols = {"Raw Attribute", "Cleaned attribute_name", "Assigned Target attribute_id"}
    if not isinstance(mapping_df, pd.DataFrame) or not required_cols.issubset(mapping_df.columns):
        return []

    # canonical attribute_description + unit come from the attribute
    # controlled vocabulary (ds_attr), keyed by attribute_id.
    vocab = {}
    try:
        vocab = {
            str(r["attribute_id"]): (
                clean(r.get("attribute_description")),
                normalize_unit(r.get("unit")),
            )
            for _, r in ds_attr.iterrows()
        }
    except NameError:
        pass

    specs = []
    for _, rec in mapping_df.iterrows():
        raw_attr = clean(rec.get("Raw Attribute"))
        attr_name = clean(rec.get("Cleaned attribute_name"))
        target_attr_id = clean(rec.get("Assigned Target attribute_id"))

        if not raw_attr or not target_attr_id:
            continue

        vocab_desc, vocab_unit = vocab.get(str(target_attr_id), (None, None))

        specs.append({
            "raw_attribute": str(raw_attr),
            # emit the human-readable cleaned name (unmapped stage), not the id
            "attribute_name": str(attr_name) if attr_name else str(target_attr_id),
            "attribute_description": vocab_desc,
            "unit": vocab_unit,
        })

    # De-duplicate by attribute name, keep first mapping row.
    deduped = {}
    for spec in specs:
        deduped.setdefault(spec["attribute_name"], spec)

    return list(deduped.values())


def get_mapped_raw_value(row, raw_attribute, input_carriers, output_carriers):
    # This row in mapping is conceptual; derive from parsed carriers.
    if raw_attribute == "Input, Output for Overall Efficiency":
        combined = input_carriers + output_carriers
        return combined if combined else None

    return clean(row.get(raw_attribute))


def map_technology(row):

    currency = row['Currency']    #!
    cap_unit = infer_main_output_unit(    #!
        clean(row.get("main_output")),      #!
        split_csv(row.get("carriers_out")),     #!
        split_csv(row.get("units_out_ratios")),     #!
    )

    result = {
        "technology_name": (
            row.get("tech_id", "")  #!
            .replace("_", " ")
        ),        

        "technology": {
    "technology_description": clean(row.get("description")),    #!
            "technology_type": clean(row.get("tech_type")),     #!
            "technology_category": clean(row.get("technology_class")),  #!
            "technology_assumption": None,
            "process_name": clean(row.get("unit_operation")),   #!
            "process_type": None,   #!
            "process_category": clean(row.get("tech_category")),    #!
            "process_assumption": None, #!
        },

        "scope": {
            "geographic_scope_description":
                clean(row.get("cost_base")),    #!
            "temporal_scope_description":   #!
                str(row.get("tech_year"))
                if not is_nan(row.get("tech_year"))
                else None,
            "capacity_scope_description":
                clean(row.get("min_installation_size")),
            "system_boundary_description":
                clean(row.get("tech_boundary")),
            "scope_assumption":
                clean(row.get("tech_maturity"))
        },

        "sources": [],
        "attributes": [],
        "balancing": {
            "inputs": [],
            "main_input": clean(row.get("main_input")),     #!
            "outputs": [],
            "main_output": clean(row.get("main_output")),      #!
            "balance_assumption": None,
        },
        "tags": {
            "related_project": "reFuel.ch",
            "tags": []
        }
    }

    attributes = result["attributes"]


    # -----------------------
    # Inputs and outputs for balancing
    # -----------------------
    input_carriers = split_csv(row.get("carriers_in"))  #!
    input_shares = split_csv_float(row.get("ratios_in"))    #!
    input_units = split_csv(row.get("units_in_ratios"))     #!

    output_carriers = split_csv(row.get("carriers_out"))    #!
    output_shares = split_csv_float(row.get("ratios_out"))  #!
    output_units = split_csv(row.get("units_out_ratios"))   #!

    result["balancing"]["inputs"] = to_balance_list(
        build_balancing_entries(
            input_carriers,
            input_shares,
            input_units,
        )
    )
    result["balancing"]["outputs"] = to_balance_list(
        build_balancing_entries(
            output_carriers,
            output_shares,
            output_units,
        )
    )

    main_output_carrier = clean(row.get("main_output")) #!
    main_output_unit = infer_main_output_unit(
        main_output_carrier,
        output_carriers,
        output_units,
    )

    # -----------------------
    # Attributes from mapping CSV only
    # -----------------------
    specs = build_attribute_specs(df_attr_mapping)
    log_main_output_unit_resolution(
        result.get("technology_name"),
        main_output_carrier,
        main_output_unit,
        specs,
    )
    for spec in specs:
        value = get_mapped_raw_value(
            row,
            spec["raw_attribute"],
            input_carriers,
            output_carriers,
        )

        resolved_unit = resolve_attribute_unit(
            spec["attribute_name"],
            spec["unit"],
            main_output_unit,
        )

        add_value(
            attributes,
            spec["attribute_name"],
            spec["attribute_description"],
            value,
            unit=resolved_unit,
        )

    # -----------------------
    # Tags
    # -----------------------
    tags = result["tags"]["tags"]

    for field in [
        "main_sector",
        "main_category",
        "category_spec",
        "tech_type",
        "tech_maturity"
    ]:
        value = clean(row.get(field))
        if value:
            tags.append(str(value))

    return prune_empty_keys(result)

In [17]:
# LAYER 2 (OVERRIDE): compatibility helpers for updated reFuel structure.
# This cell wraps map_technology from Layer 1 and adapts renamed columns.

def _row_get(row, *keys):
    for key in keys:
        if key in row and not is_nan(row.get(key)):
            return row.get(key)
    return None


def _normalize_row_aliases(row):
    """Add legacy aliases expected by Layer 1 mapper when source columns were renamed."""
    row2 = row.copy()

    alias_map = {
        "Currency": "currency",
        "Cost Base": "cost_base",
        "cost_base_year": "tech_year",
        "main_out": "main_output",
        "units_in": "units_in_ratios",
        "units_out": "units_out_ratios",
        "main_category": "tech_category",
        "category_spec": "technology_class",
        "overall_efficiency": "technical_efficiency",
        "capex_one_time_eur": "capex_one_time",
        "capex_power_capacity_eur_per_kw": "capex_per_capacity",
        "opex_one_time_eur": "opex_one_time",
        "opex_fix_power_capacity_eur_per_kw_yr": "opex_per_capacity_yr",
        "opex_energy_eur_per_unit": "opex_per_energy",
        "Min Installation Size": "min_installation_size",
        "Nominal Capacity": "nominal_capacity",
    }

    for legacy_key, new_key in alias_map.items():
        if legacy_key not in row2 or is_nan(row2.get(legacy_key)):
            if new_key in row2 and not is_nan(row2.get(new_key)):
                row2[legacy_key] = row2.get(new_key)

    return row2


def get_mapped_raw_value(row, raw_attribute, input_carriers, output_carriers):
    """Override with aliases for updated ConvTech raw attribute names."""
    if raw_attribute == "Input, Output for Overall Efficiency":
        combined = input_carriers + output_carriers
        return combined if combined else None

    alias_map = {
        "overall_efficiency": "technical_efficiency",
        "capex_one_time_eur": "capex_one_time",
        "capex_power_capacity_eur_per_kw": "capex_per_capacity",
        "opex_one_time_eur": "opex_one_time",
        "opex_fix_power_capacity_eur_per_kw_yr": "opex_per_capacity_yr",
        "opex_energy_eur_per_unit": "opex_per_energy",
    }

    value = clean(_row_get(row, raw_attribute, alias_map.get(raw_attribute, "")))
    return value


def apply_currency_to_units(record, row):
    """Replace CUR placeholder in attribute units with row-level currency."""
    currency = clean(_row_get(row, "Currency", "currency"))
    if not isinstance(currency, str):
        return record

    currency = currency.strip()
    if not currency:
        return record

    attr_list = record.get("attributes")
    if not isinstance(attr_list, list):
        attr_list = record.get("values", [])

    for item in attr_list:
        unit = item.get("unit")
        if isinstance(unit, str) and "CUR" in unit:
            item["unit"] = unit.replace("CUR", currency)

    return record


# Keep wrapper idempotent across re-runs by retaining a stable base function.
_current_map_technology = map_technology
if getattr(_current_map_technology, "_is_wrapper", False):
    _base_map_technology = _current_map_technology._base_map_technology
else:
    _base_map_technology = _current_map_technology

In [18]:
def get_src_attrs(source_text):
    if not source_text or not source_text.strip():
        return []

    # Collect: source_id -> set of linked attributes
    source_to_attrs: dict[str, set] = defaultdict(set)

    # Split on ";" to get individual attribute-group : source(s) segments
    segments = re.split(r";", source_text)

    for segment in segments:
        segment = segment.strip().rstrip(";").strip()
        if not segment:
            continue

        # Split on ":" — left side = quoted attribute names, right side = source IDs
        parts = segment.split(":", 1)
        if len(parts) != 2:
            continue

        attrs_part, sources_part = parts[0].strip(), parts[1].strip()

        # Extract attribute names (quoted strings)
        attributes = re.findall(r'"([^"]+)"', attrs_part)
        attributes = [a.strip() for a in attributes if a.strip()]

        # Extract source IDs (comma-separated, unquoted identifiers)
        # Remove any trailing punctuation or whitespace
        raw_sources = [s.strip().strip('"').strip() for s in sources_part.split(",")]
        source_ids = [s for s in raw_sources if s and s.lower() != "missing"]

        for source_id in source_ids:
            source_to_attrs[source_id].update(attributes)
    
    return source_to_attrs

In [19]:
def parse_source_text(source_text: str) -> list[dict]:
    """
    Parse a raw source_text string of the form:
        "attr1", "attr2": source_id_A; "attr3": source_id_B, source_id_C;
    
    Returns a list of source dicts following the unmapped_record schema:
        [
            {
                "source_description": "source_id_A",
                "source_type": "other",
                "link": "",
                "linked_attribute": ["attr1", "attr2"]
            },
            ...
        ]
    
    Notes:
    - A single attribute can be linked to multiple source IDs (comma-separated after the colon).
    - Duplicate source IDs across segments are merged: their linked_attributes are unioned.
    - Source IDs equal to "missing" are skipped.
    """
    
    source_to_attrs = get_src_attrs(source_text)

    # Build the sources array
    sources = []
    for source_id, linked_attrs in source_to_attrs.items():
        src_name = map_src_id.get(source_id, '')

        ## get source data from `ds_src`
        df = ds_src[ds_src['source_name']==src_name]
        if len(df) == 0:
            print(f'Cannot find source_id "{source_id}" in source dataset')
        elif len(df) > 1:
            print(f'Multiple entries found for source_id "{source_id}" in source dataset; using the first match.')
        else:
            di_src = df.iloc[0].to_dict()

            # new schema does not include source_type and link
            sources.append({
                "source_name": di_src['source_name'],
                "source_description": di_src['description'],
                "linked_attribute": sorted(linked_attrs),
            })

    return sources


def add_sources_to_record(record: dict, source_text: str) -> dict:
    """
    Parse source_text and inject a 'sources' key into the record.

    Args:
        record:      The unmapped_record dict (modified in-place and returned).
        source_text: Raw string from row.get("list_of_source_id").

    Returns:
        The same record dict with 'sources' populated.
    """
    if source_text == source_text:      # skip if source_text is NaN
        record["sources"] = parse_source_text(source_text)
    return record

In [20]:
def map_technology(row):
    """Wrapper: normalize row aliases, build record, and apply unit post-processing."""
    row_norm = _normalize_row_aliases(row)
    rec = _base_map_technology(row_norm)
    rec = apply_currency_to_units(rec, row_norm)

    ## add source 
    source_text = row.get("list_of_source_id", "")
    rec = add_sources_to_record(rec, source_text)

    return rec


map_technology._is_wrapper = True
map_technology._base_map_technology = _base_map_technology

In [21]:
def prepare_convtech_df(df_raw):
    """Normalize ConvTech table where row 1 contains machine-friendly headers."""
    df = df_raw.copy()
    header_idx = 1
    df.columns = [str(c).strip() for c in df.loc[header_idx]]
    df = df.loc[header_idx + 1:].reset_index(drop=True)

    # Remove section/header-like rows
    if "tech_id" in df.columns:
        df = df[df["tech_id"].notna()]
        df = df[df["tech_id"].astype(str).str.strip().str.lower() != "nan"]
        df = df[df["tech_id"].astype(str).str.strip().str.lower() != "tech_id"]
    if "unit_operation" in df.columns:
        df = df[df["unit_operation"].notna()]

    return df.reset_index(drop=True)

### check the content of records

In [22]:
## run record creating for a testing row

# load the data from ConvTech sheet for testing
df_conv = prepare_convtech_df(refuel_data["ConvTech"])

row = df_conv.loc[0]

# run record mapping for the testing row
record = map_technology(row)

In [23]:
# quick check: new balancing payload
record.get("balancing")

{'inputs': [{'carrier_name': 'NH3', 'share': 1.0, 'unit': 'kWh'}],
 'main_input': 'NH3',
 'outputs': [{'carrier_name': 'El13', 'share': 0.58, 'unit': 'kWh'}],
 'main_output': 'El13'}

In [24]:
# quick check: attributes come only from mapping CSV
mapped_attr_names = set(df_attr_mapping["Cleaned attribute_name"].dropna().astype(str).str.strip())
record_attr_entries = record.get("attributes", record.get("values", []))
record_attr_names = {v["attribute_name"] for v in record_attr_entries if isinstance(v, dict) and "attribute_name" in v}

print("record attributes:", len(record_attr_names))
print("not in mapping:", sorted(record_attr_names - mapped_attr_names))

record attributes: 11
not in mapping: []


In [25]:
# quick check: no CUR placeholder should remain after currency replacement
record_attr_entries = record.get("attributes", record.get("values", []))
cur_units = [
    v.get("unit")
    for v in record_attr_entries
    if isinstance(v, dict) and isinstance(v.get("unit"), str) and "CUR" in v.get("unit")
]

print("remaining CUR units:", cur_units)
print("row currency:", row.get("currency", row.get("Currency")))

remaining CUR units: []
row currency: EUR


In [26]:
record.get("attributes", record.get("values", []))

[{'attribute_name': 'technical_efficiency',
  'attribute_description': 'Overall energy conversion efficiency: main output energy / main input energy.',
  'value': '0.58'},
 {'attribute_name': 'technology_readiness_level',
  'attribute_description': 'Technology Readiness Level on the standard 1–9 scale. For non-mature technologies; TRL values progress across tech_year.',
  'value': 9},
 {'attribute_name': 'tech_maturity',
  'attribute_description': 'Aggregated maturity classification derived from TRL. Updated per tech_year.',
  'value': 'Mature'},
 {'attribute_name': 'operating_temperature',
  'attribute_description': 'Typical operating temperature of the process.',
  'unit': '°C',
  'value': 0},
 {'attribute_name': 'power_capacity_min',
  'attribute_description': 'Smallest assumed unit capacity for the technology.',
  'unit': 'CAP',
  'value': 0},
 {'attribute_name': 'lifetime_economic',
  'attribute_description': 'Economic lifetime of the asset.',
  'unit': 'yr',
  'value': 35},
 {'at

In [27]:
## check of sources
print(f'tech name: {record.get("technology_name")}')
record.get("sources", [])

tech name: NH3 CCGT El


[{'source_name': 'Power-to-Ammonia Renewable Systems Report',
  'source_description': 'Power-to-ammonia in future North European 100 % renewable power and heat system',
  'linked_attribute': ['capex_per_capacity',
   'lifetime_yr',
   'opex_per_capacity_yr',
   'opex_per_energy',
   'technical_efficiency']},
 {'source_name': 'Reaction Input-Output Stoichiometric Conversion',
  'source_description': 'Calculating Input Output Ratios of Reactions',
  'linked_attribute': ['ratios_in', 'ratios_out']},
 {'source_name': 'Danish Energy Agency Technology Dataset 2025',
  'source_description': 'Report from Danish Energy Agency',
  'linked_attribute': ['operating_temperature_c']}]

In [28]:
row['list_of_source_id']

'"capex_per_capacity", "opex_per_capacity_yr", "technical_efficiency", "opex_per_energy", "lifetime_yr": report_power_to_ammonia_2018; "ratios_in", "ratios_out": conversion_parametrization; "operating_temperature_c": report_danish_energy_2025; "ratios_in", "ratios_out": conversion_parametrization;  '

## 3.2 Validate the format of record
check if the content in record match the schema, if there is missing setion or not...

in each the record

In [29]:
# Validate whether one record matches key schema expectations.
# This is written to support future user-provided records too.

def _is_missing(value):
    if value is None:
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False


def _collect_required_unit_attributes(attr_mapping_df):
    """Return attribute names whose unit is defined in mapping and should be present in record."""
    if not isinstance(attr_mapping_df, pd.DataFrame):
        return set()

    required = set()
    required_cols = {"attribute_name", "unit"}
    if not required_cols.issubset(attr_mapping_df.columns):
        return required

    for _, rec in attr_mapping_df.iterrows():
        name = rec.get("attribute_name")
        unit = rec.get("unit")
        if _is_missing(name) or _is_missing(unit):
            continue
        unit_text = str(unit).strip()
        if unit_text != "-":
            required.add(str(name).strip())
    return required


def validate_record_against_schema(record_obj, schema_obj=None, attr_mapping_df=None):
    """Validate record with practical rules derived from current schema and mapping.

    Rules checked:
    1) Required top-level sections exist.
    2) Required nested info exists (technology/scope/balancing minimum keys).
    3) attributes items have required keys (attribute_name, value).
    4) Unit is present when attribute unit is required by mapping.
    5) balancing items have carrier_name and share.
    """
    errors = []
    warnings = []

    if not isinstance(record_obj, dict):
        return {
            "valid": False,
            "errors": ["record must be a dictionary"],
            "warnings": [],
        }

    # Rule 1: required top-level keys
    required_top = [
        "technology_name",
        "technology",
        "scope",
        "attributes",
        "balancing",
    ]
    for key in required_top:
        if key not in record_obj:
            errors.append(f"missing top-level key: {key}")

    # Rule 2: required nested keys
    tech_obj = record_obj.get("technology", {})
    if isinstance(tech_obj, dict):
        for key in ["technology_type", "process_name"]:
            if _is_missing(tech_obj.get(key)):
                errors.append(f"missing technology.{key}")
    else:
        errors.append("technology must be a dictionary")

    scope_obj = record_obj.get("scope", {})
    if isinstance(scope_obj, dict):
        if _is_missing(scope_obj.get("geographic_scope_description")):
            warnings.append("scope.geographic_scope_description is empty")
    else:
        errors.append("scope must be a dictionary")

    bal_obj = record_obj.get("balancing", {})
    if isinstance(bal_obj, dict):
        if "inputs" not in bal_obj and "outputs" not in bal_obj:
            errors.append("balancing must have inputs and/or outputs")
        for side in ["inputs", "outputs"]:
            side_items = bal_obj.get(side, [])
            if side_items is None:
                side_items = []
            if not isinstance(side_items, list):
                errors.append(f"balancing.{side} must be a list")
                continue
            for i, item in enumerate(side_items):
                if not isinstance(item, dict):
                    errors.append(f"balancing.{side}[{i}] must be a dict")
                    continue
                if _is_missing(item.get("carrier_name")):
                    errors.append(f"balancing.{side}[{i}].carrier_name is missing")
                if _is_missing(item.get("share")):
                    errors.append(f"balancing.{side}[{i}].share is missing")
    else:
        errors.append("balancing must be a dictionary")

    # Rule 3 + 4: values structure + required unit when mapping says so
    required_unit_attrs = _collect_required_unit_attributes(attr_mapping_df)
    values_obj = record_obj.get("attributes", [])
    if not isinstance(values_obj, list):
        errors.append("attributes must be a list")
    else:
        seen_attr_names = set()
        for i, item in enumerate(values_obj):
            if not isinstance(item, dict):
                errors.append(f"attributes[{i}] must be a dict")
                continue

            attr_name = item.get("attribute_name")
            attr_value = item.get("value")
            attr_unit = item.get("unit")

            if _is_missing(attr_name):
                errors.append(f"attributes[{i}].attribute_name is missing")
                continue

            attr_name = str(attr_name).strip()
            seen_attr_names.add(attr_name)

            if attr_value is None:
                errors.append(f"attributes[{i}].value is missing for {attr_name}")

            if attr_name in required_unit_attrs and _is_missing(attr_unit):
                errors.append(f"attributes[{i}].unit is required for {attr_name}")

        if len(seen_attr_names) == 0:
            warnings.append("attribute list is empty")

    # Optional Rule 5: alignment with schema keys if schema is available
    if isinstance(schema_obj, dict):
        if "unmapped_entity" not in schema_obj:
            warnings.append("schema does not contain unmapped_entity section")

    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "warnings": warnings,
    }

In [30]:
## load the data from ConvTech sheet for testing
df_conv = prepare_convtech_df(refuel_data["ConvTech"])

row = df_conv.loc[15]

## run record mapping for the testing row
record = map_technology(row)

## run record validation
validation_result = validate_record_against_schema(
    record_obj=record,
    schema_obj=schema_motel,
    attr_mapping_df=df_attr_mapping,
)

## show validation results
print("is_valid:", validation_result["valid"] )
print("error_count:", len(validation_result["errors"]))
print("warning_count:", len(validation_result["warnings"]))

if validation_result["errors"]:
    print("\nErrors:")
    for msg in validation_result["errors"][:20]:
        print("-", msg)

if validation_result["warnings"]:
    print("\nWarnings:")
    for msg in validation_result["warnings"][:20]:
        print("-", msg)

is_valid: True
error_count: 0
warning_count: 0


## 3.3 Convert all data to unmapped_records

In [31]:
## load the data from ConvTech sheet for testing

records = []

df_conv = prepare_convtech_df(refuel_data["ConvTech"])

for indx in df_conv.index[:]:
    row = df_conv.loc[indx]

    ## run record mapping for the testing row
    record = map_technology(row)

    validation_result = validate_record_against_schema(
        record_obj=record,
        schema_obj=schema_motel,
        attr_mapping_df=df_attr_mapping,
    )

    # save validation results
    record['validation_check'] = str(validation_result["valid"])
    if validation_result["valid"]:
        record['validation_note'] = f'erros: {len(validation_result["errors"])}; warnings: {len(validation_result["warnings"])}'

    records.append(record)

    ## show validation results
    print(f'technology_name: {record.get("technology_name")}, indx={indx}')

    if validation_result["errors"]:
        print("\nErrors:")
        for msg in validation_result["errors"][:20]:
            print("-", msg)

    if validation_result["warnings"]:
        print("\nWarnings:")
        for msg in validation_result["warnings"][:20]:
            print("-", msg)   

technology_name: NH3 CCGT El, indx=0
technology_name: NH3 OCGT El, indx=1
technology_name: CH4 CCGT El, indx=2
technology_name: CH4 CHP ElTh, indx=3
technology_name: CH4 FuelCell El, indx=4
technology_name: CH4 FuelCell ElBdg, indx=5
technology_name: CH4 OCGT El, indx=6
technology_name: Geothermal CHP ElTh, indx=7
technology_name: H2 CCGT El, indx=8
technology_name: H2 FuelCell El, indx=9
technology_name: H2 FuelCell El, indx=10
technology_name: H2 FuelCell ElBdg, indx=11
technology_name: H2 FuelCell ElBdg, indx=12
technology_name: H2 OCGT El, indx=13
technology_name: El PumpedHydro Pump, indx=14
technology_name: Hydro PumpedTurbine El, indx=15
technology_name: Hydro RunOfRiver El, indx=16
technology_name: Hydro Reservoir El, indx=17
technology_name: MSW Incineration El, indx=18
technology_name: Solar PVAlpine El, indx=19
technology_name: Solar PVAlpine El, indx=20
technology_name: Solar PVAlpine El, indx=21
technology_name: Solar PVAlpine El, indx=22
technology_name: Solar PVRooftop E

In [32]:
## export records to a yaml file
from pathlib import Path

path_export = 'motel-db/unmapped_entity/refuel.yaml'
Path(path_export).parent.mkdir(parents=True, exist_ok=True)
with open(path_export, 'w', encoding='utf-8') as f:
    yaml.dump(records, f, sort_keys=False, allow_unicode=True)

In [33]:
## test to load the yaml file

with open(path_export, 'r', encoding='utf-8') as f:
    loaded_records = yaml.safe_load(f)

loaded_records

[{'technology_name': 'NH3 CCGT El',
  'technology': {'technology_type': 'Conversion',
   'technology_category': 'Gas turbine',
   'process_name': 'CCGT'},
  'scope': {'geographic_scope_description': 'ECA',
   'temporal_scope_description': '2050',
   'capacity_scope_description': 0,
   'system_boundary_description': 'Plant ready to operate',
   'scope_assumption': 'Mature'},
  'attributes': [{'attribute_name': 'technical_efficiency',
    'attribute_description': 'Overall energy conversion efficiency: main output energy / main input energy.',
    'value': '0.58'},
   {'attribute_name': 'technology_readiness_level',
    'attribute_description': 'Technology Readiness Level on the standard 1–9 scale. For non-mature technologies; TRL values progress across tech_year.',
    'value': 9},
   {'attribute_name': 'tech_maturity',
    'attribute_description': 'Aggregated maturity classification derived from TRL. Updated per tech_year.',
    'value': 'Mature'},
   {'attribute_name': 'operating_tempe

In [34]:
len(loaded_records)

99

In [35]:
loaded_records[0]

{'technology_name': 'NH3 CCGT El',
 'technology': {'technology_type': 'Conversion',
  'technology_category': 'Gas turbine',
  'process_name': 'CCGT'},
 'scope': {'geographic_scope_description': 'ECA',
  'temporal_scope_description': '2050',
  'capacity_scope_description': 0,
  'system_boundary_description': 'Plant ready to operate',
  'scope_assumption': 'Mature'},
 'attributes': [{'attribute_name': 'technical_efficiency',
   'attribute_description': 'Overall energy conversion efficiency: main output energy / main input energy.',
   'value': '0.58'},
  {'attribute_name': 'technology_readiness_level',
   'attribute_description': 'Technology Readiness Level on the standard 1–9 scale. For non-mature technologies; TRL values progress across tech_year.',
   'value': 9},
  {'attribute_name': 'tech_maturity',
   'attribute_description': 'Aggregated maturity classification derived from TRL. Updated per tech_year.',
   'value': 'Mature'},
  {'attribute_name': 'operating_temperature',
   'attrib